In [1]:
# Cell 1 — SPEI-3 Water Scarcity: user inputs, hazard definition, and run identifiers
# Pattern matches the Extreme Heat notebook, adapted for monthly SPEI-3.
#
# User inputs:
#   - ISO3: country code
#   - AS_OF_DATE: reference end date (YYYY-MM-DD)
#
# Hazard definition (SPEI-3):
#   - Metric: standardised precipitation evapotranspiration index, 3-month accumulation (SPEI-3)
#   - Time window: last 12 months up to AS_OF_DATE (monthly data)
#   - No baseline-relative threshold needed (SPEI already standardised)
#   - Thresholds (dryness / scarcity): SPEI <= -1.0, -1.5, -2.0
#   - “Population affected” means: population in cells where SPEI-3 is below threshold
#     at least once within the 12-month window (any-month exceedance).
#
# Note: CDS requests will use AOI bbox from admin2 union (later cell) via 'area' parameter.

from __future__ import annotations

from pathlib import Path
from datetime import date
import json
import pandas as pd

# ---- USER INPUTS ----
ISO3 = "SDN"  # <- set country ISO3
AS_OF_DATE = "2025-11-30"  # <- set reference date (end of window), YYYY-MM-DD

# ---- WINDOW (monthly) ----
# Define the last-12-month window as inclusive end month, and previous 11 months.
END_MONTH = pd.to_datetime(AS_OF_DATE).to_period("M")
WINDOW_MONTHS = [END_MONTH - i for i in range(0, 12)][::-1]
WINDOW_START_DATE = (WINDOW_MONTHS[0].to_timestamp(how="start")).date().isoformat()
WINDOW_END_DATE = (WINDOW_MONTHS[-1].to_timestamp(how="end")).date().isoformat()

# ---- HAZARD THRESHOLDS (SPEI-3) ----
# Standard drought/scarcity severity classes (negative SPEI = drier than normal).
SPEI_THRESHOLDS = {
    "spei3_le_-1_0": -1.0,  # moderate dryness
    "spei3_le_-1_5": -1.5,  # severe dryness
    "spei3_le_-2_0": -2.0,  # extreme dryness
}

SPEI_VAR = "SPEI3"

# Default threshold for QC plots/reporting
DEFAULT_THRESHOLD_KEY = "spei3_le_-1_5"

# ---- RUN IDENTIFIER ----
# Keep run ids stable and informative; include ISO3 + as_of + window start/end.
RUN_ID = f"{ISO3}_{WINDOW_START_DATE}_{WINDOW_END_DATE}_spei3"

print("ISO3:", ISO3)
print("AS_OF_DATE:", AS_OF_DATE)
print(
    "Monthly window:",
    f"{WINDOW_START_DATE} → {WINDOW_END_DATE}",
    f"({len(WINDOW_MONTHS)} months)",
)
print("SPEI thresholds:", SPEI_THRESHOLDS)
print("Run ID:", RUN_ID)

# ---- Persist minimal run config (useful for QC PDF cover later) ----
run_config = {
    "iso3": ISO3,
    "as_of_date": AS_OF_DATE,
    "window_start_date": WINDOW_START_DATE,
    "window_end_date": WINDOW_END_DATE,
    "window_months": [str(p) for p in WINDOW_MONTHS],
    "hazard": "water_scarcity_spei3",
    "thresholds": SPEI_THRESHOLDS,
    "default_threshold": DEFAULT_THRESHOLD_KEY,
    "notes": {
        "metric": "SPEI-3 (3-month accumulation), monthly",
        "exposure_rule": "any-month threshold exceedance within the last 12 months",
        "baseline_relative_threshold": False,
        "consecutive_months_required": False,
    },
}

# Set these now; file paths will be defined in the next cell when we create DIRS
RUN_METADATA = {"run_config": run_config}

ISO3: SDN
AS_OF_DATE: 2025-11-30
Monthly window: 2024-12-01 → 2025-11-30 (12 months)
SPEI thresholds: {'spei3_le_-1_0': -1.0, 'spei3_le_-1_5': -1.5, 'spei3_le_-2_0': -2.0}
Run ID: SDN_2024-12-01_2025-11-30_spei3


In [2]:
# Cell 2 — Directory structure + run metadata scaffolding (SPEI-3)
# Mirrors the Extreme Heat notebook: one per-run output folder + shared baseline cache.

from pathlib import Path
from datetime import datetime
import json

# ---- Base paths (edit if needed) ----
# Keep outputs outside repo root if you prefer; this matches your current pattern.
OUT_ROOT = Path("../outputs/water_scarcity_spei3_admin2").resolve()

# COD GDB (admin2 single source of truth)
COD_GDB_ZIP = Path("../COD/global_admin_boundaries_matched_latest.gdb.zip").resolve()
COD_LAYER = "admin2"

# WorldPop path will be hardcoded per country later (per your workflow)
WORLDPOP_PATH = None  # placeholder (set in Cell 4)


# ---- Helper ----
def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p


# ---- Run folder structure ----
RUN_DIR = ensure_dir(OUT_ROOT / ISO3 / RUN_ID)

DIRS = {
    "base": RUN_DIR,
    "raw_cds": ensure_dir(RUN_DIR / "cds_raw"),  # downloaded zips
    "raw_extracted": ensure_dir(RUN_DIR / "cds_raw" / "extracted"),  # extracted NetCDFs
    "masks_native": ensure_dir(RUN_DIR / "masks_native"),  # SPEI grid masks
    "masks_worldpop": ensure_dir(RUN_DIR / "masks_worldpop"),  # WorldPop grid masks
    "pop_affected": ensure_dir(RUN_DIR / "pop_affected"),  # pop-masked rasters
    "tables": ensure_dir(RUN_DIR / "tables"),  # final CSV(s)
    "qc": ensure_dir(RUN_DIR / "qc"),
    "logs": ensure_dir(RUN_DIR / "logs"),
}

# ---- Shared caches (stable inputs that should not change for a given ISO3) ----
# For SPEI we don’t need a baseline reference distribution, but we DO want to avoid re-downloading
# the same historical months repeatedly when re-running.
CACHE_ROOT = Path("../outputs/_cache").resolve()
CACHE_DIRS = {
    "base": ensure_dir(CACHE_ROOT / "water_scarcity_spei3"),
    "cds_raw": ensure_dir(CACHE_ROOT / "water_scarcity_spei3" / ISO3 / "cds_raw"),
    "cds_extracted": ensure_dir(
        CACHE_ROOT / "water_scarcity_spei3" / ISO3 / "cds_raw" / "extracted"
    ),
    "aoi": ensure_dir(CACHE_ROOT / "water_scarcity_spei3" / ISO3 / "aoi"),
}

# ---- Run metadata helpers ----
RUN_METADATA_PATH = RUN_DIR / "run_metadata.json"


def update_run_metadata_artifact(key: str, path: Path, note: str | None = None) -> None:
    RUN_METADATA.setdefault("artifacts", {})
    RUN_METADATA["artifacts"][key] = {"path": str(path), "note": note}
    RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))


# Initialise metadata file
RUN_METADATA["paths"] = {
    "out_root": str(OUT_ROOT),
    "run_dir": str(RUN_DIR),
    "cod_gdb_zip": str(COD_GDB_ZIP),
    "cod_layer": COD_LAYER,
    "cache_root": str(CACHE_ROOT),
}
RUN_METADATA["created_utc"] = datetime.utcnow().isoformat(timespec="seconds") + "Z"
RUN_METADATA["run_id"] = RUN_ID

RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

print("Run directory:", RUN_DIR)
print("DIRS keys:", list(DIRS.keys()))
print("CACHE_DIRS keys:", list(CACHE_DIRS.keys()))
print("Run metadata:", RUN_METADATA_PATH)

Run directory: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/outputs/water_scarcity_spei3_admin2/SDN/SDN_2024-12-01_2025-11-30_spei3
DIRS keys: ['base', 'raw_cds', 'raw_extracted', 'masks_native', 'masks_worldpop', 'pop_affected', 'tables', 'qc', 'logs']
CACHE_DIRS keys: ['base', 'cds_raw', 'cds_extracted', 'aoi']
Run metadata: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/outputs/water_scarcity_spei3_admin2/SDN/SDN_2024-12-01_2025-11-30_spei3/run_metadata.json


In [3]:
# Cell 3 — Load admin2 boundaries (single source of truth) and filter to ISO3 (SPEI-3)

import geopandas as gpd

if not COD_GDB_ZIP.exists():
    raise FileNotFoundError(f"COD GDB zip not found: {COD_GDB_ZIP}")

# Read admin2 layer (authoritative)
admin2_gdf = gpd.read_file(str(COD_GDB_ZIP), layer=COD_LAYER)

# Basic schema checks
required_cols = {"iso3", "adm2_pcode", "geometry"}
missing = required_cols - set(admin2_gdf.columns)
if missing:
    raise KeyError(
        f"Missing required columns in COD admin2: {missing}. Available: {list(admin2_gdf.columns)}"
    )

# Filter to country
admin2_gdf = admin2_gdf[admin2_gdf["iso3"] == ISO3].copy()
admin2_gdf = admin2_gdf.dropna(subset=["adm2_pcode", "geometry"]).reset_index(drop=True)

if admin2_gdf.empty:
    raise ValueError(
        f"No admin2 features found for ISO3={ISO3} in {COD_GDB_ZIP} layer={COD_LAYER}"
    )

# Ensure valid geometries
admin2_gdf["geometry"] = admin2_gdf["geometry"].buffer(0)
admin2_gdf = admin2_gdf[admin2_gdf.is_valid].copy()

print(f"Loaded admin2 features for {ISO3}: {len(admin2_gdf):,}")
print("CRS:", admin2_gdf.crs)
print("Example adm2_pcode:", admin2_gdf["adm2_pcode"].iloc[0])

# Record artifact (optional: write a quick GeoPackage for inspection)
ADMIN2_GPKG_PATH = DIRS["qc"] / f"{ISO3}_admin2.gpkg"
admin2_gdf.to_file(ADMIN2_GPKG_PATH, layer="admin2", driver="GPKG")
update_run_metadata_artifact(
    "admin2_gpkg", ADMIN2_GPKG_PATH, "Admin2 boundaries filtered to ISO3"
)

/Users/jamesebrown/miniconda3/envs/utci-env/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP.
  return ogr_read(


Loaded admin2 features for SDN: 188
CRS: EPSG:4326
Example adm2_pcode: SD07090


In [4]:
# Cell 4 — AOI geometry + CDS bbox (area) from admin2 union + bounds hash (SPEI-3)
# The CDS 'area' parameter expects: [North, West, South, East] in EPSG:4326.

import hashlib
from shapely.ops import unary_union

# Reproject to EPSG:4326 for CDS bbox and hashing
admin2_4326 = admin2_gdf.to_crs("EPSG:4326").copy()

# Union all admin2 geometries into a single AOI polygon
country_geom_4326 = unary_union(admin2_4326.geometry)

if country_geom_4326.is_empty:
    raise RuntimeError("AOI union geometry is empty — cannot proceed.")

# Bounds in EPSG:4326
minx, miny, maxx, maxy = country_geom_4326.bounds
west, south, east, north = float(minx), float(miny), float(maxx), float(maxy)

print("Country bounds (EPSG:4326):")
print(f"  West={west:.4f}, South={south:.4f}, East={east:.4f}, North={north:.4f}")

# CDS area order: [North, West, South, East]
cds_area = [north, west, south, east]
print("CDS area:", cds_area)


# ---- Bounds hash helpers ----
def bounds_hash_from_area(area: list[float], precision: int = 6) -> str:
    """
    Stable hash of the CDS area array [N, W, S, E].
    Precision is important to avoid hash churn due to floating rounding noise.
    """
    rounded = [round(float(x), precision) for x in area]
    payload = ",".join(f"{x:.{precision}f}" for x in rounded).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def write_bounds_hash(area: list[float], out_path: Path, precision: int = 6) -> str:
    h = bounds_hash_from_area(area, precision=precision)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(h)
    return h


AOI_BOUNDS_HASH_PATH = CACHE_DIRS["aoi"] / f"{ISO3}_admin2_union_bounds_hash.txt"
AOI_HASH = write_bounds_hash(cds_area, AOI_BOUNDS_HASH_PATH)

print("AOI bounds hash:", AOI_HASH)
print("Hash file:", AOI_BOUNDS_HASH_PATH)

# Record to metadata
RUN_METADATA["aoi"] = {
    "cds_area": cds_area,
    "bounds_4326": {"west": west, "south": south, "east": east, "north": north},
    "bounds_hash": AOI_HASH,
    "bounds_hash_path": str(AOI_BOUNDS_HASH_PATH),
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))
update_run_metadata_artifact(
    "aoi_bounds_hash",
    AOI_BOUNDS_HASH_PATH,
    "Hash for admin2-union AOI bounds (CDS area)",
)

Country bounds (EPSG:4326):
  West=21.8146, South=8.6376, East=38.5800, North=23.1469
CDS area: [23.146886230000007, 21.81455583500002, 8.637558893000005, 38.58003616299999]
AOI bounds hash: e8b1a5b62f8c3b0203802cf516336fbec7fc482b9e2b9a0e81909b1722e53e7a
Hash file: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/outputs/_cache/water_scarcity_spei3/SDN/aoi/SDN_admin2_union_bounds_hash.txt


In [5]:
# Cell 5 — WorldPop raster (hardcoded per country) + validity mask (SPEI-3)
# As with the extreme heat notebook, we treat WorldPop as the aggregation grid.
# IMPORTANT: exclude NoData and negative values from all sums.

import rasterio
import numpy as np

# ---- Hardcode WorldPop path per country (update as needed) ----
# Example: Sudan (SDN) — adjust for your own filenames/years.
WORLDPOP_PATH = Path(
    "cds_spei_data/population/sdn_pop_2025_CN_100m_R2025A_v1.tif"
).resolve()

if not WORLDPOP_PATH.exists():
    raise FileNotFoundError(
        f"WorldPop raster not found: {WORLDPOP_PATH}\n"
        "Set WORLDPOP_PATH to your country-clipped WorldPop GeoTIFF."
    )

with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float64")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata

print("WorldPop path:", WORLDPOP_PATH)
print("WorldPop CRS:", wp_crs)
print("WorldPop shape:", wp_shape)
print("WorldPop nodata:", wp_nodata)
print("WorldPop min/max:", float(np.nanmin(wp_arr)), float(np.nanmax(wp_arr)))
print("WorldPop negatives:", int((wp_arr < 0).sum()))

# Valid population pixels: finite, not nodata, and >= 0
pop_valid_mask = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid_mask &= wp_arr != float(wp_nodata)
pop_valid_mask &= wp_arr >= 0

total_pop = float(np.nansum(wp_arr[pop_valid_mask]))
print(f"Total population (valid WorldPop pixels): {total_pop:,.0f}")

# Record in metadata
RUN_METADATA["worldpop"] = {
    "path": str(WORLDPOP_PATH),
    "crs": str(wp_crs),
    "shape": [int(wp_shape[0]), int(wp_shape[1])],
    "nodata": wp_nodata,
    "total_pop_valid": total_pop,
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))
update_run_metadata_artifact(
    "worldpop_raster",
    WORLDPOP_PATH,
    "Country-clipped WorldPop GeoTIFF (aggregation grid)",
)

WorldPop path: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/ERA5-DROUGHT/cds_spei_data/population/sdn_pop_2025_CN_100m_R2025A_v1.tif
WorldPop CRS: EPSG:4326
WorldPop shape: (16252, 20438)
WorldPop nodata: -99999.0
WorldPop min/max: -99999.0 524.4417724609375
WorldPop negatives: 323788616
Total population (valid WorldPop pixels): 50,849,352


In [7]:
# QC — WorldPop extent covers admin2 country extent (bounds-only, no pixel values)

import rasterio
import numpy as np

# admin2_gdf is already filtered to ISO3
# Ensure admin2 geometry is valid (optional but helps avoid odd bounds)
admin2 = admin2_gdf.copy()
admin2["geometry"] = admin2.geometry.buffer(0)

with rasterio.open(WORLDPOP_PATH) as src:
    ras_crs = src.crs
    ras_bounds = src.bounds  # left, bottom, right, top

# Project admin2 to raster CRS and get bounds
admin2_ras = admin2.to_crs(ras_crs)
geom_bounds = admin2_ras.total_bounds  # minx, miny, maxx, maxy

minx, miny, maxx, maxy = geom_bounds
rminx, rminy, rmaxx, rmaxy = (
    ras_bounds.left,
    ras_bounds.bottom,
    ras_bounds.right,
    ras_bounds.top,
)

# Containment with a tiny tolerance (in CRS units; degrees for EPSG:4326)
TOL = 1e-9
covers = (
    (rminx <= minx + TOL)
    and (rminy <= miny + TOL)
    and (rmaxx >= maxx - TOL)
    and (rmaxy >= maxy - TOL)
)

# Quantify gaps (positive numbers mean admin extends beyond raster)
gap_w = max(0.0, rminx - minx)
gap_s = max(0.0, rminy - miny)
gap_e = max(0.0, maxx - rmaxx)
gap_n = max(0.0, maxy - rmaxy)

print(f"Raster CRS: {ras_crs}")
print(f"Raster bounds:   W={rminx:.6f}, S={rminy:.6f}, E={rmaxx:.6f}, N={rmaxy:.6f}")
print(f"Admin2 bounds:   W={minx:.6f}, S={miny:.6f}, E={maxx:.6f}, N={maxy:.6f}")
print(f"Covers admin2 extent? {covers}")
print(
    f"Extent gaps (units of CRS): west={gap_w:.6f}, south={gap_s:.6f}, east={gap_e:.6f}, north={gap_n:.6f}"
)

# Fail-fast if not covered
if not covers:
    raise RuntimeError(
        f"WorldPop raster extent does not fully cover admin2 country extent for {ISO3}. "
        f"Gaps: W={gap_w}, S={gap_s}, E={gap_e}, N={gap_n} (CRS units). "
    )

Raster CRS: EPSG:4326
Raster bounds:   W=21.814999, S=8.681667, E=38.846666, N=22.225000
Admin2 bounds:   W=21.814556, S=8.637559, E=38.580036, N=23.146886
Covers admin2 extent? False
Extent gaps (units of CRS): west=0.000443, south=0.044108, east=0.000000, north=0.921886


RuntimeError: WorldPop raster extent does not fully cover admin2 country extent for SDN. Gaps: W=0.0004433577399751698, S=0.04410807493998625, E=0.0, N=0.9218859829000152 (CRS units). Use an extended/mosaicked population surface.

In [9]:
# Cell 6 — CDS downloads for SPEI-3 (monthly): consolidated + intermediate fallback
# Dataset: derived-drought-historical-monthly
# Uses CDS area subset from Cell 4: cds_area = [North, West, South, East]
#
# Strategy:
#   1) Try consolidated_dataset for each month in the last-12-month window
#   2) If that month fails (e.g. not yet consolidated), try intermediate_dataset
#   3) Cache per month + dataset_type so reruns are idempotent

import zipfile
import cdsapi
import pandas as pd
from pathlib import Path

# ---- CDS constants ----
DATASET_SPEI = "derived-drought-historical-monthly"
SPEI_VAR = "standardised_precipitation_evapotranspiration_index"
ACCUMULATION_PERIOD = "3"
VERSION = "1_0"
PRODUCT_TYPE = ["reanalysis"]

DATASET_TYPE_CONS = "consolidated_dataset"
DATASET_TYPE_INT = "intermediate_dataset"


def months_for_last_12(as_of: str) -> list[tuple[int, int]]:
    end = pd.to_datetime(as_of).to_period("M")
    return [(int((end - n).year), int((end - n).month)) for n in range(0, 12)][::-1]


def cds_request_spei(
    area: list[float], year: int, month: int, dataset_type: str
) -> dict:
    """
    Create a CDS request dict for derived-drought-historical-monthly.
    area must be [North, West, South, East] (EPSG:4326).
    """
    return {
        "variable": [SPEI_VAR],
        "accumulation_period": [ACCUMULATION_PERIOD],
        "version": VERSION,
        "product_type": PRODUCT_TYPE,
        "dataset_type": dataset_type,  # consolidated_dataset / intermediate_dataset
        "year": [str(year)],
        "month": [f"{month:02d}"],
        "area": area,
    }


def download_cds(dataset: str, request: dict, out_zip: Path) -> tuple[bool, str | None]:
    out_zip.parent.mkdir(parents=True, exist_ok=True)
    try:
        client = cdsapi.Client()
        result = client.retrieve(dataset, request)
        result.download(target=str(out_zip))
        return True, None
    except Exception as e:
        return False, str(e)


def extract_zip_to_dir(zip_path: Path, out_dir: Path) -> list[Path]:
    out_subdir = ensure_dir(out_dir / zip_path.stem)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_subdir)
    return sorted(out_subdir.rglob("*.nc"))


def ensure_downloads(manifest: list[dict], kind: str) -> None:
    mf_path = DIRS["logs"] / f"cds_manifest_{kind}.csv"
    pd.DataFrame(manifest).to_csv(mf_path, index=False)
    update_run_metadata_artifact(
        f"cds_manifest_{kind}", mf_path, f"{len(manifest)} requests recorded"
    )

    ok_count = sum(1 for r in manifest if r.get("ok"))
    if ok_count == 0:
        errs = [r.get("error") for r in manifest if r.get("error")]
        raise RuntimeError(
            f"All CDS downloads failed for '{kind}'. Example error: {errs[0] if errs else 'Unknown'}"
        )


# ---- Cache key: AOI hash + window end month ----
END_YYYYMM = pd.to_datetime(AS_OF_DATE).strftime("%Y%m")
CACHE_KEY = f"{ISO3}_spei3_{AOI_HASH}_{END_YYYYMM}"

CDS_CACHE_ZIP_DIR = ensure_dir(CACHE_DIRS["cds_raw"])
CDS_CACHE_EXTRACT_DIR = ensure_dir(CACHE_DIRS["cds_extracted"])

# ---- Download month-by-month with fallback ----
months = months_for_last_12(AS_OF_DATE)
manifest = []

for y, m in months:
    # 1) consolidated attempt (cached)
    cons_zip = CDS_CACHE_ZIP_DIR / f"{CACHE_KEY}_{y}{m:02d}_{DATASET_TYPE_CONS}.zip"
    if cons_zip.exists() and cons_zip.stat().st_size > 0:
        manifest.append(
            {
                "year": y,
                "month": f"{m:02d}",
                "dataset_type": DATASET_TYPE_CONS,
                "ok": True,
                "error": None,
                "path": str(cons_zip),
                "cached": True,
            }
        )
        continue

    req_cons = cds_request_spei(cds_area, y, m, DATASET_TYPE_CONS)
    ok, err = download_cds(DATASET_SPEI, req_cons, cons_zip)
    manifest.append(
        {
            "year": y,
            "month": f"{m:02d}",
            "dataset_type": DATASET_TYPE_CONS,
            "ok": ok,
            "error": err,
            "path": str(cons_zip),
            "cached": False,
        }
    )

    # 2) fallback to intermediate if consolidated failed
    if not ok:
        int_zip = CDS_CACHE_ZIP_DIR / f"{CACHE_KEY}_{y}{m:02d}_{DATASET_TYPE_INT}.zip"
        if int_zip.exists() and int_zip.stat().st_size > 0:
            manifest.append(
                {
                    "year": y,
                    "month": f"{m:02d}",
                    "dataset_type": DATASET_TYPE_INT,
                    "ok": True,
                    "error": None,
                    "path": str(int_zip),
                    "cached": True,
                    "fallback_for": str(cons_zip),
                }
            )
            continue

        req_int = cds_request_spei(cds_area, y, m, DATASET_TYPE_INT)
        ok2, err2 = download_cds(DATASET_SPEI, req_int, int_zip)
        manifest.append(
            {
                "year": y,
                "month": f"{m:02d}",
                "dataset_type": DATASET_TYPE_INT,
                "ok": ok2,
                "error": err2,
                "path": str(int_zip),
                "cached": False,
                "fallback_for": str(cons_zip),
            }
        )

ensure_downloads(manifest, "spei_recent")

# ---- Extract NetCDFs (idempotent per zip) ----
nc_paths: list[Path] = []
for row in manifest:
    if not row.get("ok"):
        continue
    zpath = Path(row["path"])
    if not zpath.exists() or zpath.stat().st_size == 0:
        continue

    target_dir = CDS_CACHE_EXTRACT_DIR / zpath.stem
    if target_dir.exists() and any(target_dir.rglob("*.nc")):
        nc_paths.extend(sorted(target_dir.rglob("*.nc")))
    else:
        nc_paths.extend(extract_zip_to_dir(zpath, CDS_CACHE_EXTRACT_DIR))

nc_paths = sorted({p.resolve() for p in nc_paths})
if not nc_paths:
    raise RuntimeError("SPEI ZIPs extracted but no .nc files found.")

# ---- Summarise coverage ----
ok_months = {
    (r["year"], r["month"])
    for r in manifest
    if r.get("ok") and r.get("dataset_type") in (DATASET_TYPE_CONS, DATASET_TYPE_INT)
}
missing = [(y, f"{m:02d}") for (y, m) in months if (y, f"{m:02d}") not in ok_months]
if missing:
    print("WARNING: missing months after fallback:", missing)

print(f"SPEI NC files: {len(nc_paths)}")
print("Example NC:", nc_paths[0])

RUN_METADATA["cds_downloads_spei"] = {
    "dataset": DATASET_SPEI,
    "variable": SPEI_VAR,
    "accumulation_period": ACCUMULATION_PERIOD,
    "aoi_hash": AOI_HASH,
    "end_yyyymm": END_YYYYMM,
    "n_requests": len(manifest),
    "n_nc": len(nc_paths),
    "cache_zip_dir": str(CDS_CACHE_ZIP_DIR),
    "cache_extract_dir": str(CDS_CACHE_EXTRACT_DIR),
    "dataset_types": {"primary": DATASET_TYPE_CONS, "fallback": DATASET_TYPE_INT},
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

SPEI NC files: 12
Example NC: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/outputs/_cache/water_scarcity_spei3/SDN/cds_raw/extracted/SDN_spei3_e8b1a5b62f8c3b0203802cf516336fbec7fc482b9e2b9a0e81909b1722e53e7a_202511_202412_consolidated_dataset/SPEI3_genlogistic_global_era5_moda_ref1991to2020_202412.area-subset.23.146886230000007.38.58003616299999.8.637558893000005.21.81455583500002.nc


4915

In [10]:
# Cell 7 — Open SPEI NetCDFs, build a single monthly DataArray for the 12-month window (patched)

import xarray as xr
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401
from shapely.geometry import mapping


def find_spei_var(ds: xr.Dataset) -> str:
    if SPEI_VAR in ds.data_vars:
        return SPEI_VAR
    candidates = [v for v in ds.data_vars if v not in ("crs",)]
    if not candidates:
        raise KeyError(f"No data variables found in dataset. Vars={list(ds.data_vars)}")
    return candidates[0]


# ---- Load all NetCDFs and collect DataArrays ----
das = []
for p in nc_paths:
    ds = xr.open_dataset(p, decode_times=True)
    vname = find_spei_var(ds)
    da = ds[vname]

    # Standardise dim names (CDS sometimes uses latitude/longitude)
    rename = {}
    if "latitude" in da.dims:
        rename["latitude"] = "lat"
    if "longitude" in da.dims:
        rename["longitude"] = "lon"
    if rename:
        da = da.rename(rename)

    for d in ("time", "lat", "lon"):
        if d not in da.dims:
            raise KeyError(f"Expected dim '{d}' not found in {p.name}. Dims={da.dims}")

    das.append(da)

# ---- Concatenate and de-duplicate ----
spei = xr.concat(das, dim="time").sortby("time")

tvals = spei["time"].values
_, idx = np.unique(tvals, return_index=True)
spei = spei.isel(time=np.sort(idx)).sortby("time")

# ---- Ensure rioxarray spatial metadata survives concat ----
# This is the key fix for MissingSpatialDimensionError after concat.
spei = spei.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
spei = spei.rio.write_crs("EPSG:4326", inplace=False)

# ---- Subset to the required 12 months ----
target_months = [
    str(p) for p in WINDOW_MONTHS
]  # ordered list like ["2024-12", ..., "2025-11"]
spei_months = spei["time"].dt.strftime("%Y-%m").values.tolist()

sel = np.isin(spei_months, target_months)
spei = spei.isel(time=sel).sortby("time")

n_months = int(spei.sizes.get("time", 0))
print(f"SPEI months loaded: {n_months} (expected 12)")

if n_months == 0:
    available = sorted(set(pd.to_datetime(tvals).astype("datetime64[M]").astype(str)))
    raise RuntimeError(
        "No months selected for the target 12-month window.\n"
        f"Target months: {target_months}\n"
        f"Available months in loaded NetCDFs (YYYY-MM): {available[:24]}{' ...' if len(available) > 24 else ''}"
    )

# Report missing months explicitly (helps confirm CDS request completeness)
loaded_months = set(spei["time"].dt.strftime("%Y-%m").values.tolist())
missing_months = [m for m in target_months if m not in loaded_months]
if missing_months:
    print("WARNING: Missing months from target window:", missing_months)

print(
    "First/last timestamp:",
    str(pd.to_datetime(spei["time"].values[0]).date()),
    "→",
    str(pd.to_datetime(spei["time"].values[-1]).date()),
)

# Ensure rioxarray spatial metadata exists on the final object (concat can drop it)
spei = spei.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
spei = spei.rio.write_crs("EPSG:4326", inplace=False)

# Clip to country polygon (admin2 union)
geoms = [mapping(country_geom_4326)]
spei = spei.rio.clip(geoms, crs="EPSG:4326", drop=True, all_touched=True)
spei_da = spei.astype("float32")

RUN_METADATA["spei_data"] = {
    "n_months": n_months,
    "time_start": str(pd.to_datetime(spei_da["time"].values[0]).date()),
    "time_end": str(pd.to_datetime(spei_da["time"].values[-1]).date()),
    "dims": {k: int(v) for k, v in spei_da.sizes.items()},
    "missing_months": missing_months,
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

print("SPEI DataArray ready:", spei_da)

SPEI months loaded: 12 (expected 12)
First/last timestamp: 2024-12-01 → 2025-11-01
SPEI DataArray ready: <xarray.DataArray 'SPEI3' (time: 12, lat: 58, lon: 67)> Size: 187kB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, n

In [12]:
# QC — quantify NaNs and confirm we have data inside the clipped AOI

import numpy as np

total = spei_da.size
nan_count = int(np.isnan(spei_da.values).sum())
finite_count = int(np.isfinite(spei_da.values).sum())

print(f"Total cells (time*lat*lon): {total:,}")
print(f"NaNs: {nan_count:,} ({nan_count/total:.1%})")
print(f"Finite: {finite_count:,} ({finite_count/total:.1%})")

# Check at least some finite data exists each month
finite_by_month = (
    np.isfinite(spei_da.values).reshape(spei_da.sizes["time"], -1).sum(axis=1)
)
print("Finite cells per month:", finite_by_month.tolist())

# Summary stats over finite values only (across all months)
vals = spei_da.values[np.isfinite(spei_da.values)]
print(
    "Finite SPEI3 min/median/max:",
    float(np.min(vals)),
    float(np.median(vals)),
    float(np.max(vals)),
)


Total cells (time*lat*lon): 46,632
NaNs: 14,652 (31.4%)
Finite: 31,980 (68.6%)
Finite cells per month: [2665, 2665, 2665, 2665, 2665, 2665, 2665, 2665, 2665, 2665, 2665, 2665]
Finite SPEI3 min/median/max: -14.788824081420898 -0.7785202264785767 4.162806987762451


In [13]:
# Cell 8 — SPEI masks + reproject to WorldPop grid + population affected rasters
# Output artifacts (GeoTIFFs on WorldPop grid):
#   - masks_worldpop/spei_mask_<key>.tif            (uint8 0/1)
#   - pop_exposed/spei_pop_affected_<key>.tif       (float32, WorldPop counts where mask==1 else 0)
#
# SPEI is already a standardised index (relative), so no baseline needed.
# We use simple thresholding across the 12-month window (any month meeting threshold counts as affected).

import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin

# --- Thresholds (adjust if you want different cut points) ---
# Convention: more negative = drier (worse drought)
SPEI_THRESHOLDS = {
    "rel_spei_le_m1p0": -1.0,
    "rel_spei_le_m1p5": -1.5,
    "rel_spei_le_m2p0": -2.0,
}

# Default threshold for QC/plots later
DEFAULT_SPEI_KEY = "rel_spei_le_m1p5"

# --- Ensure output dirs exist (match your DIRS keys) ---
MASKS_NATIVE_DIR = ensure_dir(DIRS["masks_native"] / "spei")
MASKS_WP_DIR = ensure_dir(DIRS["masks_worldpop"] / "spei")
POP_AFFECTED_DIR = ensure_dir(DIRS["pop_affected"] / "spei")
QC_DIR = ensure_dir(DIRS["qc"] / "spei_masks")


def spei_any_month_mask(spei_da: xr.DataArray, thr: float) -> xr.DataArray:
    """
    Returns a 2D (lat, lon) boolean mask: True if ANY month in the window has SPEI <= thr.
    NaNs remain False (i.e., treated as not affected) — consistent with conservative masking.
    """
    cond = spei_da <= thr
    cond = cond.fillna(False)
    return cond.any(dim="time")


def write_bool_geotiff_from_da(path: Path, mask_da_2d: xr.DataArray) -> None:
    """
    Write a 2D boolean mask (lat, lon) to a GeoTIFF using its own grid (EPSG:4326).
    """
    if not set(mask_da_2d.dims) >= {"lat", "lon"}:
        raise ValueError(
            f"Expected mask dims include ('lat','lon'), got {mask_da_2d.dims}"
        )

    arr = mask_da_2d.values.astype(np.uint8)

    lats = mask_da_2d["lat"].values
    lons = mask_da_2d["lon"].values

    # Assume regular grid (CDS lat/lon)
    dlat = float(np.abs(lats[1] - lats[0]))
    dlon = float(np.abs(lons[1] - lons[0]))

    # lat is typically descending in CDS; compute north edge robustly
    north = float(np.max(lats) + dlat / 2.0)
    west = float(np.min(lons) - dlon / 2.0)

    transform = from_origin(west, north, dlon, dlat)

    path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=arr.shape[0],
        width=arr.shape[1],
        count=1,
        dtype="uint8",
        crs="EPSG:4326",
        transform=transform,
        nodata=0,
        compress="deflate",
        predictor=2,
    ) as dst:
        dst.write(arr, 1)


def reproject_mask_to_worldpop(
    mask_da_2d: xr.DataArray, wp_profile: dict
) -> np.ndarray:
    """
    Reproject a 2D mask DataArray (lat/lon) to the WorldPop grid (nearest).
    Returns uint8 mask array on WorldPop shape.
    """
    # Prepare a temporary source raster in memory-like form (we'll use rasterio.reproject)
    src_arr = mask_da_2d.values.astype(np.uint8)

    lats = mask_da_2d["lat"].values
    lons = mask_da_2d["lon"].values
    dlat = float(np.abs(lats[1] - lats[0]))
    dlon = float(np.abs(lons[1] - lons[0]))
    north = float(np.max(lats) + dlat / 2.0)
    west = float(np.min(lons) - dlon / 2.0)
    src_transform = from_origin(west, north, dlon, dlat)

    dst_arr = np.zeros((wp_profile["height"], wp_profile["width"]), dtype=np.uint8)

    from rasterio.warp import reproject, Resampling

    reproject(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_transform,
        src_crs="EPSG:4326",
        dst_transform=wp_profile["transform"],
        dst_crs=wp_profile["crs"],
        resampling=Resampling.nearest,
        src_nodata=0,
        dst_nodata=0,
    )
    return dst_arr


def write_uint8_geotiff(
    path: Path, arr: np.ndarray, wp_profile: dict, nodata: int = 0
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    profile = wp_profile.copy()
    profile.update(
        dtype="uint8",
        count=1,
        nodata=nodata,
        compress="deflate",
        predictor=2,
    )
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr.astype(np.uint8), 1)


def write_float32_geotiff(
    path: Path, arr: np.ndarray, wp_profile: dict, nodata: float = 0.0
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    profile = wp_profile.copy()
    profile.update(
        dtype="float32",
        count=1,
        nodata=nodata,
        compress="deflate",
        predictor=2,
    )
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr.astype(np.float32), 1)


# --- Build WorldPop profile for writing outputs ---
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_profile = wp.profile
    wp_profile.update(count=1)  # single-band outputs
    wp_arr = wp.read(1).astype("float64")
    wp_nodata = wp.nodata

# Rebuild validity mask exactly as before
pop_valid_mask = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid_mask &= wp_arr != float(wp_nodata)
pop_valid_mask &= wp_arr >= 0

# --- Compute and write masks for each threshold ---
mask_products = {}

for key, thr in SPEI_THRESHOLDS.items():
    # 1) native mask on SPEI grid
    mask_native_da = spei_any_month_mask(spei_da, thr)

    native_path = MASKS_NATIVE_DIR / f"spei_mask_{key}_native.tif"
    write_bool_geotiff_from_da(native_path, mask_native_da)

    # 2) reproject to WorldPop grid
    mask_wp = reproject_mask_to_worldpop(mask_native_da, wp_profile)
    mask_wp_path = MASKS_WP_DIR / f"spei_mask_{key}_worldpop.tif"
    write_uint8_geotiff(mask_wp_path, mask_wp, wp_profile, nodata=0)

    # 3) population affected on WorldPop grid
    pop_affected = np.zeros_like(wp_arr, dtype=np.float32)
    affected = (mask_wp == 1) & pop_valid_mask
    pop_affected[affected] = wp_arr[affected].astype(np.float32)

    pop_path = POP_AFFECTED_DIR / f"spei_pop_affected_{key}.tif"
    write_float32_geotiff(pop_path, pop_affected, wp_profile, nodata=0.0)

    # simple QC numbers
    pop_sum = float(np.nansum(pop_affected))
    mask_products[key] = {
        "threshold": float(thr),
        "mask_native_path": str(native_path),
        "mask_worldpop_path": str(mask_wp_path),
        "pop_affected_path": str(pop_path),
        "pop_affected_sum": pop_sum,
    }
    print(f"{key}: pop affected = {pop_sum:,.0f}")

# Record to metadata
RUN_METADATA["spei_masks"] = {
    "thresholds": {k: float(v) for k, v in SPEI_THRESHOLDS.items()},
    "default_key": DEFAULT_SPEI_KEY,
    "products": mask_products,
    "window_months": [str(p) for p in WINDOW_MONTHS],
    "aggregation_rule": "any_month_spei_le_threshold",
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

# Register key artifacts (default threshold only to keep metadata tidy)
update_run_metadata_artifact(
    "spei_default_mask_worldpop",
    Path(mask_products[DEFAULT_SPEI_KEY]["mask_worldpop_path"]),
    "Default SPEI-3 drought mask on WorldPop grid (uint8 0/1)",
)
update_run_metadata_artifact(
    "spei_default_pop_affected",
    Path(mask_products[DEFAULT_SPEI_KEY]["pop_affected_path"]),
    "Default SPEI-3 population affected raster on WorldPop grid (float32 counts)",
)

rel_spei_le_m1p0: pop affected = 50,616,152
rel_spei_le_m1p5: pop affected = 48,891,092
rel_spei_le_m2p0: pop affected = 47,609,456


In [14]:
# Cell 9 — Admin2 aggregation: % population affected by SPEI thresholds
# Produces: <ISO3>_admin2_water_scarcity_spei3_<AS_OF_DATE>.csv
#
# Uses WorldPop grid as common raster:
#   - WORLDPOP_PATH (country-clipped WorldPop)
#   - POP_AFFECTED_DIR / spei_pop_affected_<key>.tif (from Cell 8)
#
# Output columns:
#   adm2_pcode, pop_total,
#   pop_affected_<key>, pct_affected_<key>  for each threshold key

import pandas as pd
import rasterio
from rasterio.features import rasterize
import numpy as np

# ---- Load admin2 boundaries (filtered earlier to ISO3) ----
admin2 = admin2_gdf.copy()

if "adm2_pcode" not in admin2.columns:
    raise KeyError(f"'adm2_pcode' not found. Available columns: {list(admin2.columns)}")

admin2 = (
    admin2[["adm2_pcode", "geometry"]]
    .dropna(subset=["adm2_pcode", "geometry"])
    .reset_index(drop=True)
)

# ---- Open WorldPop as reference grid ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float64")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata

# Robust validity mask (exclude nodata + negatives)
pop_valid_mask = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid_mask &= wp_arr != float(wp_nodata)
pop_valid_mask &= wp_arr >= 0

# Reproject admin2 to match WorldPop CRS
admin2_wp = admin2.to_crs(wp_crs)

# ---- Rasterise admin2 polygons to unique IDs ----
shapes = [(geom, i + 1) for i, geom in enumerate(admin2_wp.geometry)]
admin2_id_raster = rasterize(
    shapes=shapes,
    out_shape=wp_shape,
    transform=wp_transform,
    fill=0,
    dtype="int32",
    all_touched=True,
)

flat_ids = admin2_id_raster.ravel()
flat_pop = wp_arr.ravel()
flat_pop_valid = pop_valid_mask.ravel()

valid = (flat_ids > 0) & flat_pop_valid
ids_valid = flat_ids[valid]
pop_valid = flat_pop[valid]

# Total pop by admin2 id (denominator)
pop_total_by_id = np.bincount(
    ids_valid, weights=pop_valid, minlength=len(admin2_wp) + 1
)

# Scaffold output table
out = pd.DataFrame(
    {
        "adm2_pcode": admin2_wp["adm2_pcode"].values,
        "admin2_id": np.arange(1, len(admin2_wp) + 1, dtype=int),
    }
)
out["pop_total"] = out["admin2_id"].map(
    lambda i: float(pop_total_by_id[i]) if i < len(pop_total_by_id) else 0.0
)

# ---- Aggregate affected population per threshold ----
for key in SPEI_THRESHOLDS.keys():
    pop_path = POP_AFFECTED_DIR / f"spei_pop_affected_{key}.tif"
    if not pop_path.exists():
        raise FileNotFoundError(f"Missing pop affected raster for {key}: {pop_path}")

    with rasterio.open(pop_path) as src:
        arr = src.read(1).astype("float64")
        arr_nodata = src.nodata

        # Grid safety check
        if arr.shape != wp_shape:
            raise ValueError(
                f"Shape mismatch for {key}: {arr.shape} vs WorldPop {wp_shape}"
            )

        arr_ok = np.isfinite(arr)
        if arr_nodata is not None:
            arr_ok &= arr != float(arr_nodata)
        arr_ok &= arr >= 0

        flat_arr = arr.ravel()
        valid_arr = valid & arr_ok.ravel()

        ids_valid_arr = flat_ids[valid_arr]
        vals_valid_arr = flat_arr[valid_arr]

        pop_aff_by_id = np.bincount(
            ids_valid_arr, weights=vals_valid_arr, minlength=len(admin2_wp) + 1
        )

    col_pop = f"pop_affected_{key}"
    col_pct = f"pct_affected_{key}"

    out[col_pop] = out["admin2_id"].map(
        lambda i: float(pop_aff_by_id[i]) if i < len(pop_aff_by_id) else 0.0
    )
    out[col_pct] = np.where(
        out["pop_total"] > 0, (out[col_pop] / out["pop_total"]) * 100.0, np.nan
    )

# ---- Save table ----
OUT_CSV_PATH = DIRS["tables"] / f"{ISO3}_admin2_water_scarcity_spei3_{AS_OF_DATE}.csv"
out = out.drop(columns=["admin2_id"])
out.to_csv(OUT_CSV_PATH, index=False)

update_run_metadata_artifact(
    "admin2_water_scarcity_table",
    OUT_CSV_PATH,
    "Admin2 population % affected by water scarcity (SPEI-3 thresholds; any month in 12-month window)",
)

print("Wrote:", OUT_CSV_PATH)
print(out.head(10).to_string(index=False))

RUN_METADATA["admin2_table_spei"] = {
    "path": str(OUT_CSV_PATH),
    "n_admin2": int(len(out)),
    "thresholds": {k: float(v) for k, v in SPEI_THRESHOLDS.items()},
    "default_key": DEFAULT_SPEI_KEY,
    "window_months": [str(p) for p in WINDOW_MONTHS],
    "rule": "any_month_spei_le_threshold",
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

Wrote: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/outputs/water_scarcity_spei3_admin2/SDN/SDN_2024-12-01_2025-11-30_spei3/tables/SDN_admin2_water_scarcity_spei3_2025-11-30.csv
adm2_pcode     pop_total  pop_affected_rel_spei_le_m1p0  pct_affected_rel_spei_le_m1p0  pop_affected_rel_spei_le_m1p5  pct_affected_rel_spei_le_m1p5  pop_affected_rel_spei_le_m2p0  pct_affected_rel_spei_le_m2p0
   SD07090 124069.470488                  124069.470488                     100.000000                  124069.470488                     100.000000                  124069.470488                     100.000000
   SD16008 138245.265524                   91563.087760                      66.232350                   20687.166528                      14.964105                    1030.134316                       0.745150
   SD14037 271009.943144                  271009.943144                     100.000000                  271009.943144                     100.000000             

9797

In [15]:
# Cell 10 — QC + visualisations (MEMORY-SAFE: windowed sums + downsampled plots)
# FIX: mask WorldPop nodata/negatives in plots so colour scale is meaningful.

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling
from rasterio.plot import plotting_extent
import matplotlib.pyplot as plt
from pathlib import Path

QC_SPEI_DIR = ensure_dir(DIRS["qc"] / "spei")
FIG_DIR = ensure_dir(QC_SPEI_DIR / "figures")

mask_wp_path = MASKS_WP_DIR / f"spei_mask_{DEFAULT_SPEI_KEY}_worldpop.tif"
pop_aff_path = POP_AFFECTED_DIR / f"spei_pop_affected_{DEFAULT_SPEI_KEY}.tif"

if not mask_wp_path.exists():
    raise FileNotFoundError(mask_wp_path)
if not pop_aff_path.exists():
    raise FileNotFoundError(pop_aff_path)


def windowed_sum_worldpop(path: Path) -> tuple[float, dict]:
    """Sum valid WorldPop values using block windows (excludes nodata and negatives)."""
    total = 0.0
    with rasterio.open(path) as src:
        nodata = src.nodata
        profile = src.profile
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window).astype("float64", copy=False)
            m = np.isfinite(arr)
            if nodata is not None:
                m &= arr != float(nodata)
            m &= arr >= 0
            if m.any():
                total += float(arr[m].sum())
    return total, profile


def windowed_sum_masked_pop(worldpop_path: Path, mask_path: Path) -> float:
    """Sum WorldPop where mask==1, using block windows (constant memory)."""
    total = 0.0
    with rasterio.open(worldpop_path) as wp, rasterio.open(mask_path) as ms:
        if (wp.height != ms.height) or (wp.width != ms.width):
            raise ValueError("WorldPop and mask shapes differ.")
        if wp.crs != ms.crs:
            raise ValueError("WorldPop and mask CRS differ.")
        if not wp.transform.almost_equals(ms.transform):
            raise ValueError("WorldPop and mask transforms differ.")

        wp_nodata = wp.nodata
        mask_nodata = ms.nodata

        for _, window in wp.block_windows(1):
            pop = wp.read(1, window=window).astype("float64", copy=False)
            mask = ms.read(1, window=window).astype("uint8", copy=False)

            pop_ok = np.isfinite(pop)
            if wp_nodata is not None:
                pop_ok &= pop != float(wp_nodata)
            pop_ok &= pop >= 0

            mask_ok = mask == 1
            if mask_nodata is not None:
                mask_ok &= mask != int(mask_nodata)

            sel = pop_ok & mask_ok
            if sel.any():
                total += float(pop[sel].sum())
    return total


def windowed_sum_pop_affected(pop_aff_path: Path) -> float:
    """Sum affected raster values using block windows (excludes nodata and negatives)."""
    total = 0.0
    with rasterio.open(pop_aff_path) as src:
        nodata = src.nodata
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window).astype("float64", copy=False)
            m = np.isfinite(arr)
            if nodata is not None:
                m &= arr != float(nodata)
            m &= arr >= 0
            if m.any():
                total += float(arr[m].sum())
    return total


def read_overview(path: Path, max_dim: int = 2000, resampling=Resampling.nearest):
    """
    Read a downsampled overview for plotting.
    Returns (arr, extent, meta).
    """
    with rasterio.open(path) as src:
        h, w = src.height, src.width
        scale = max(h / max_dim, w / max_dim, 1.0)
        out_h = int(np.ceil(h / scale))
        out_w = int(np.ceil(w / scale))

        arr = src.read(
            1,
            out_shape=(out_h, out_w),
            resampling=resampling,
        )

        transform = src.transform * src.transform.scale((w / out_w), (h / out_h))
        extent = plotting_extent(arr, transform)
        meta = {
            "crs": src.crs,
            "transform": transform,
            "nodata": src.nodata,
            "shape": (out_h, out_w),
        }
    return arr, extent, meta


# ---- Sanitisation for plotting (avoid nodata dominating colour scale) ----
def sanitise_worldpop_for_plot(arr: np.ndarray, nodata) -> np.ndarray:
    a = arr.astype("float64", copy=False)
    bad = ~np.isfinite(a)
    if nodata is not None:
        bad |= a == float(nodata)
    bad |= a < 0
    a = a.copy()
    a[bad] = np.nan
    return a


def sanitise_pop_affected_for_plot(arr: np.ndarray, nodata) -> np.ndarray:
    a = arr.astype("float64", copy=False)
    bad = ~np.isfinite(a)
    if nodata is not None:
        bad |= a == float(nodata)
    bad |= a < 0
    a = a.copy()
    a[bad] = np.nan
    return a


def sanitise_mask_for_plot(arr: np.ndarray) -> np.ndarray:
    # keep only 0/1; everything else -> nan so colourbar isn't weird
    a = arr.astype("float64", copy=False)
    bad = ~np.isin(a, [0, 1])
    a = a.copy()
    a[bad] = np.nan
    return a


# ---- Admin2 boundaries ----
admin2 = admin2_gdf.copy()
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_crs = wp.crs
admin2_wp = admin2.to_crs(wp_crs)

# Simplify for speed
admin2_wp_simpl = admin2_wp.copy()
admin2_wp_simpl["geometry"] = admin2_wp_simpl.geometry.simplify(
    tolerance=0.002, preserve_topology=True
)

# ---- QC metrics (constant memory) ----
worldpop_total_pop, wp_profile = windowed_sum_worldpop(WORLDPOP_PATH)
mask_pop_sum = windowed_sum_masked_pop(WORLDPOP_PATH, mask_wp_path)
pop_affected_sum = windowed_sum_pop_affected(pop_aff_path)

with rasterio.open(WORLDPOP_PATH) as wp, rasterio.open(
    mask_wp_path
) as ms, rasterio.open(pop_aff_path) as pa:
    grid_ok = (
        (wp.height == ms.height == pa.height)
        and (wp.width == ms.width == pa.width)
        and (wp.crs == ms.crs == pa.crs)
        and wp.transform.almost_equals(ms.transform)
        and wp.transform.almost_equals(pa.transform)
    )
    wp_nodata = wp.nodata
    pop_aff_nodata = pa.nodata
    wp_shape = (wp.height, wp.width)

qc = {
    "iso3": ISO3,
    "as_of_date": AS_OF_DATE,
    "window_months": ", ".join([str(p) for p in WINDOW_MONTHS]),
    "default_threshold_key": DEFAULT_SPEI_KEY,
    "default_threshold_value": float(SPEI_THRESHOLDS[DEFAULT_SPEI_KEY]),
    "worldpop_total_pop": worldpop_total_pop,
    "mask_pop_sum": mask_pop_sum,
    "pop_affected_sum": pop_affected_sum,
    "mask_vs_pop_affected_diff": (mask_pop_sum - pop_affected_sum),
    "grid_alignment_ok": bool(grid_ok),
    "worldpop_shape": str(wp_shape),
    "worldpop_crs": str(wp_crs),
    "worldpop_nodata": wp_nodata,
    "pop_aff_nodata": pop_aff_nodata,
}
qc_df = pd.DataFrame([qc])
QC_CSV = QC_SPEI_DIR / f"{ISO3}_spei_qc_{AS_OF_DATE}.csv"
qc_df.to_csv(QC_CSV, index=False)
update_run_metadata_artifact(
    "spei_qc_table", QC_CSV, "QC summary for SPEI water scarcity processing"
)
print(qc_df.T.to_string(header=False))

# ---- FIG 1: WorldPop overview ----
wp_over, wp_extent, wp_meta = read_overview(
    WORLDPOP_PATH, max_dim=2000, resampling=Resampling.average
)
wp_over = sanitise_worldpop_for_plot(wp_over, wp_meta["nodata"])

fig1, ax1 = plt.subplots(figsize=(8, 6))
im1 = ax1.imshow(wp_over, extent=wp_extent, origin="upper")
admin2_wp_simpl.boundary.plot(ax=ax1, linewidth=0.3, edgecolor="black")
ax1.set_title(f"{ISO3} WorldPop overview with Admin2 outlines")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
FIG1 = FIG_DIR / f"{ISO3}_fig1_worldpop_{AS_OF_DATE}.png"
fig1.savefig(FIG1, dpi=200, bbox_inches="tight")
plt.close(fig1)

# ---- FIG 2: SPEI mask overview ----
mask_over, mask_extent, _ = read_overview(
    mask_wp_path, max_dim=2000, resampling=Resampling.nearest
)
mask_over = sanitise_mask_for_plot(mask_over)

fig2, ax2 = plt.subplots(figsize=(8, 6))
im2 = ax2.imshow(mask_over, extent=mask_extent, origin="upper")
admin2_wp_simpl.boundary.plot(ax=ax2, linewidth=0.3, edgecolor="black")
ax2.set_title(f"{ISO3} SPEI-3 mask ({DEFAULT_SPEI_KEY}) overview (WorldPop grid)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
FIG2 = FIG_DIR / f"{ISO3}_fig2_spei_mask_{DEFAULT_SPEI_KEY}_{AS_OF_DATE}.png"
fig2.savefig(FIG2, dpi=200, bbox_inches="tight")
plt.close(fig2)

# ---- FIG 3: Pop affected overview ----
pa_over, pa_extent, pa_meta = read_overview(
    pop_aff_path, max_dim=2000, resampling=Resampling.average
)
pa_over = sanitise_pop_affected_for_plot(pa_over, pa_meta["nodata"])

fig3, ax3 = plt.subplots(figsize=(8, 6))
im3 = ax3.imshow(pa_over, extent=pa_extent, origin="upper")
admin2_wp_simpl.boundary.plot(ax=ax3, linewidth=0.3, edgecolor="black")
ax3.set_title(f"{ISO3} Population affected overview (SPEI-3 {DEFAULT_SPEI_KEY})")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
FIG3 = FIG_DIR / f"{ISO3}_fig3_pop_affected_{DEFAULT_SPEI_KEY}_{AS_OF_DATE}.png"
fig3.savefig(FIG3, dpi=200, bbox_inches="tight")
plt.close(fig3)

# ---- FIG 4: Admin2 choropleth (% affected) ----
pct_col = f"pct_affected_{DEFAULT_SPEI_KEY}"
admin2_join = admin2_wp.merge(
    out[["adm2_pcode", "pop_total", pct_col]], on="adm2_pcode", how="left"
)

fig4, ax4 = plt.subplots(figsize=(8, 6))
admin2_join.plot(column=pct_col, ax=ax4, legend=True)
admin2_wp_simpl.boundary.plot(ax=ax4, linewidth=0.3, edgecolor="black")
ax4.set_title(f"{ISO3} Admin2 % population affected (SPEI-3 {DEFAULT_SPEI_KEY})")
ax4.set_axis_off()
FIG4 = FIG_DIR / f"{ISO3}_fig4_admin2_pct_{DEFAULT_SPEI_KEY}_{AS_OF_DATE}.png"
fig4.savefig(FIG4, dpi=200, bbox_inches="tight")
plt.close(fig4)

RUN_METADATA["spei_qc_figures"] = {
    "qc_csv": str(QC_CSV),
    "fig_worldpop": str(FIG1),
    "fig_mask": str(FIG2),
    "fig_pop_affected": str(FIG3),
    "fig_admin2_choropleth": str(FIG4),
}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

print("Saved figures:")
print(" -", FIG1)
print(" -", FIG2)
print(" -", FIG3)
print(" -", FIG4)

iso3                                                                                                                              SDN
as_of_date                                                                                                                 2025-11-30
window_months              2024-12, 2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11
default_threshold_key                                                                                                rel_spei_le_m1p5
default_threshold_value                                                                                                          -1.5
worldpop_total_pop                                                                                                    50849351.702969
mask_pop_sum                                                                                                          48891090.671971
pop_affected_sum                                              

In [16]:
# Cell 11 — Export QC + figures to a single PDF (SPEI water scarcity)
# Uses matplotlib PdfPages for a simple, dependable multi-page report.

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

PDF_PATH = QC_SPEI_DIR / f"{ISO3}_spei_qc_report_{AS_OF_DATE}.pdf"

# Re-load QC table for printing into PDF
qc_df = pd.read_csv(QC_CSV)

with PdfPages(PDF_PATH) as pdf:
    # Page 1: QC text summary
    fig, ax = plt.subplots(figsize=(8.27, 11.69))  # A4 portrait-ish
    ax.axis("off")

    lines = []
    lines.append(f"SPEI-3 Water Scarcity QC Report — {ISO3}")
    lines.append(f"As-of date: {AS_OF_DATE}")
    lines.append(f"Window months: {qc_df.loc[0,'window_months']}")
    lines.append("")
    lines.append(
        f"Default threshold: {qc_df.loc[0,'default_threshold_key']}  (<= {qc_df.loc[0,'default_threshold_value']})"
    )
    lines.append("")
    lines.append("Key checks:")
    lines.append(f"- Grid alignment OK: {bool(qc_df.loc[0,'grid_alignment_ok'])}")
    lines.append(
        f"- WorldPop total population: {qc_df.loc[0,'worldpop_total_pop']:,.0f}"
    )
    lines.append(
        f"- Masked population (WorldPop sum where mask==1): {qc_df.loc[0,'mask_pop_sum']:,.0f}"
    )
    lines.append(f"- Pop affected raster sum: {qc_df.loc[0,'pop_affected_sum']:,.0f}")
    lines.append(
        f"- Difference (mask pop - pop affected): {qc_df.loc[0,'mask_vs_pop_affected_diff']:,.0f}"
    )
    lines.append("")
    lines.append("Raster metadata:")
    lines.append(f"- WorldPop shape: {qc_df.loc[0,'worldpop_shape']}")
    lines.append(f"- WorldPop CRS: {qc_df.loc[0,'worldpop_crs']}")
    lines.append(f"- WorldPop NoData: {qc_df.loc[0,'worldpop_nodata']}")
    lines.append(f"- Pop affected NoData: {qc_df.loc[0,'pop_aff_nodata']}")

    ax.text(0.05, 0.95, "\n".join(lines), va="top", ha="left", fontsize=11)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # Pages 2–5: figures
    for fig_path in [FIG1, FIG2, FIG3, FIG4]:
        img = plt.imread(fig_path)
        fig, ax = plt.subplots(figsize=(11.69, 8.27))  # A4 landscape-ish
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(Path(fig_path).name, fontsize=10)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

update_run_metadata_artifact(
    "spei_qc_pdf", PDF_PATH, "QC report PDF for SPEI water scarcity (figures + summary)"
)

RUN_METADATA["spei_qc_report_pdf"] = {"path": str(PDF_PATH)}
RUN_METADATA_PATH.write_text(json.dumps(RUN_METADATA, indent=2))

print("Wrote PDF:", PDF_PATH)

Wrote PDF: /Users/jamesebrown/Library/Mobile Documents/com~apple~CloudDocs/Code/WIA/outputs/water_scarcity_spei3_admin2/SDN/SDN_2024-12-01_2025-11-30_spei3/qc/spei/SDN_spei_qc_report_2025-11-30.pdf
